# Data analysis for AMC chains

In [17]:
# Import libraries I may need

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Validate scope

1. Count to ensure theater had more than 10 showings total.
2. Return a list of all movie titles and verify online they have captioned versions available.
3. Count screenings per film per theater in the week and remove films that had fewer than 4 screenings during the week; these do not fall under the rule.

In [18]:
## Import .CSV

amc_screenings = pd.read_csv('/Users/jackwalker/Data/2026/CUNY_fall/adv-data-assignment-1/data/import/screenings/raw/amc/amc.csv')
print(amc_screenings)

            Date Day of week                           Film  Showtime  \
0     2026-11-02      Monday  AMC Screen Unseen: November 2   7:00 PM   
1     2026-11-02      Monday                  La Bola Negra  11:30 AM   
2     2026-11-02      Monday                  La Bola Negra  12:10 PM   
3     2026-11-02      Monday                  La Bola Negra   3:15 PM   
4     2026-11-02      Monday                  La Bola Negra   3:45 PM   
...          ...         ...                            ...       ...   
2459  2026-11-08      Sunday            Godzilla Minus Zero   9:45 PM   
2460  2026-11-08      Sunday             The Cat in the Hat   1:45 PM   
2461  2026-11-08      Sunday             The Cat in the Hat   4:40 PM   
2462  2026-11-08      Sunday             The Cat in the Hat   7:30 PM   
2463  2026-11-08      Sunday             The Cat in the Hat  10:20 PM   

     Peak hours? Open captioned?              Theater Company  \
0            Yes              No  AMC 19th St. East 6     

In [19]:
## Check for number of screenings per theater
amc_screenings['Theater'].value_counts()

Theater
AMC Empire 25                   474
AMC Lincoln Square 13           339
AMC Kips Bay 15                 269
AMC 34th Street 14              257
AMC Bay Plaza Cinema 13         197
AMC DINE-IN Staten Island 11    192
AMC Magic Johnson Harlem 9      143
AMC Fresh Meadows 7             120
AMC Village 7                   113
AMC Bay Terrace 6               109
AMC 84th Street 6               102
AMC 19th St. East 6             101
AMC Orpheum 7                    48
Name: count, dtype: int64

In [20]:
## Okay, they all have more than 10.

## Now I'm going to store each unique film title in a .CSV file and then
## manually check online to confirm it has a captioned version possible.

open_caption_available = (
    amc_screenings[['Film']]
    .drop_duplicates()
    .sort_values('Film')
    .reset_index(drop=True)
)

open_caption_available['Available?'] = pd.Series(
    pd.NA, index=open_caption_available.index, dtype='boolean'
)

## print(open_caption_available)

## From a manual check online, some are unclear on caption access

captions_missing = [
    'AMC Screen Unseen: November 2',
    'Club Kid with Q&A',
    'I Play Rocky Special Q&A with Cast/Filmmakers',
    'Rammstein - Live in Mexico City',
    "Stray Kids World Tour 'RUN IT' LIVE VIEWING",
    'The Cat in the Hat : Private Theatre Rental',
]

# Set False for those titles, True for all others
open_caption_available['Available?'] = ~open_caption_available['Film'].isin(captions_missing)

print(open_caption_available)

                                   Film  Available?
0                                 #CULT        True
1                   A Talent for Murder        True
2         AMC Screen Unseen: November 2       False
3      Aadarsha Kutumbam - House No. 47        True
4                               Breeder        True
..                                  ...         ...
57                      Wild Horse Nine        True
58              Wild at Heart - Unrated        True
59                             Wildwood        True
60               You Can See Everything        True
61  Your Mother Your Mother Your Mother        True

[62 rows x 2 columns]


In [21]:
## Remove the ones we can't confirm from our dataset; this makes our final
## result a minimum number of screenings failing to meet the mark, as 
## opposed to an overshot bc some didn't even have captions as an option.

amc_screenings_filtered = amc_screenings[
    ~amc_screenings['Film'].isin(captions_missing)
].copy()

print('Rows removed:', len(amc_screenings) - len(amc_screenings_filtered))

Original rows: 2464
Filtered rows: 2431
Rows removed: 33


In [15]:
## Remove screenings with fewer than four per theater.

## Store counts of films per theater using groupby
screenings_count = amc_screenings_filtered.groupby(['Theater', 'Film']).size()

# Find theater screenings exempt due to frequency criteria
screenings_too_few = screenings_count[screenings_count < 4].index

## Remove those screenings
amc_screenings_filtered = (
    amc_screenings_filtered
    .set_index(['Theater', 'Film'])
    .drop(index=screenings_too_few, errors='ignore')
    .reset_index()
    .copy()
)

print(amc_screenings_filtered)

                  Theater                 Film        Date Day of week  \
0     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
1     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
2     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
3     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
4     AMC 19th St. East 6        La Bola Negra  2026-11-02      Monday   
...                   ...                  ...         ...         ...   
2300        AMC Village 7  Godzilla Minus Zero  2026-11-08      Sunday   
2301        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   
2302        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   
2303        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   
2304        AMC Village 7   The Cat in the Hat  2026-11-08      Sunday   

      Showtime Peak hours? Open captioned? Company  \
0     11:30 AM          No              No     AMC   
1  